<a href="https://colab.research.google.com/github/Brygges/publicCys/blob/main/Foldseek_Tab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To get started, simply go to ```Runtime > Run all``` found at the top hotbar, after which the ```Upload``` should be available for use.

In [2]:
#@title Prepend Files
import subprocess, os
subprocess.run("wget -q https://github.com/steineggerlab/foldseek/releases/download/10-941cd33/foldseek-linux-avx2.tar.gz && tar xzf foldseek-linux-avx2.tar.gz", shell=True)
os.environ["PATH"] += ":/content/foldseek/bin"

subprocess.run("wget -q https://github.com/Brygges/publicCys/raw/main/database.tar.xz && tar xf database.tar.xz", shell=True)

CompletedProcess(args='wget -q https://github.com/Brygges/publicCys/raw/main/database.tar.xz && tar xf database.tar.xz', returncode=0)

In [18]:
#@title Upload and Search { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd, subprocess

upload = widgets.FileUpload(accept='.pdb,.mmcif,.cif', multiple=False, description="Choose file")
run_button = widgets.Button(description="Run", button_style="success", icon="search")
out = widgets.Output()

def foldseek_search(i):
  with out:
    clear_output()
    if not upload.value:
      print("No file chosen.")
      return
    filename = list(upload.value.keys())[0]
    f = upload.value[filename]

    ext = os.path.splitext(filename)[1]
    query_path = f"query{ext}"
    with open(query_path, "wb") as fh:
      fh.write(f["content"])

    result = subprocess.run(
      ["foldseek", "easy-search", query_path, "database/database", "result.m8", "tmp", "--format-output", "query,target,evalue,prob", "-e", "1e-3"],
      capture_output=True, text=True
    )
    clear_output()
    if result.returncode != 0:
      print("Error running Foldseek. Try again.")
      print(result.stderr or result.stdout)
      return

    try:
      df = pd.read_csv("result.m8", sep="\t", header=None,
                       names=["query","target","evalue","prob"])
      df["target"] = df["target"].str.replace(r"_unrelaxed.*$", "", regex=True)
      if df.empty:
        print("No matches found for given query.")
      else:
        display(df[["target","evalue","prob"]]
                .sort_values("evalue"))
    except Exception:
      print("Search finished but results could not be parsed.")

run_button.on_click(foldseek_search)
display(widgets.VBox([upload, run_button, out]))